# App1: Reaction Load Params - REST API to JSON Schema

**Workspace ID:** 2672  
**Entity ID:** 12162  
**App URL:** https://demo.viktor.ai/workspaces/2672/app/editor/12162

This notebook reads the entity data through the VIKTOR REST API and converts the saved parameters/properties to JSON Schema.


In [ ]:
import os
import json
from pathlib import Path
from typing import Any, Dict

import requests
from dotenv import find_dotenv, load_dotenv

# Force reload .env file (in case you just edited it)
DOTENV_PATH = find_dotenv(usecwd=True)
load_dotenv(DOTENV_PATH, override=True)

print(f"Loaded .env: {DOTENV_PATH or '<not found>'}")

# Hard-coded workspace and entity IDs
WORKSPACE_ID = 2672
ENTITY_ID = 12162
OUTPUT_FILE = "app1_params_schema.json"

# Accepts "demo", "demo.viktor.ai", or "https://demo.viktor.ai".
VIKTOR_ENVIRONMENT = os.getenv("VIKTOR_ENVIRONMENT", "demo")

def build_viktor_api_base_url(environment: str) -> str:
    environment = environment.strip().rstrip("/")
    if environment.startswith("https://"):
        host = environment.removeprefix("https://")
    elif environment.startswith("http://"):
        host = environment.removeprefix("http://")
    else:
        host = environment

    if not host.endswith(".viktor.ai"):
        host = f"{host}.viktor.ai"

    return f"https://{host}/api"

BASE_URL = build_viktor_api_base_url(VIKTOR_ENVIRONMENT)

print(f"Target: Workspace {WORKSPACE_ID}, Entity {ENTITY_ID}")
print(f"REST API base URL: {BASE_URL}")

## 1. Connect to VIKTOR REST API


In [ ]:
# Initialize REST API session with a Personal Access Token.
# The .env value must be ONLY the token, for example:
# VIKTOR_API_TOKEN=vktrpat_...
# Do not include "Bearer ", "Authorization:", angle brackets, or copied quotes.

def normalize_viktor_pat(raw_token: str | None, *, env_var: str = "VIKTOR_API_TOKEN") -> str:
    if raw_token is None:
        raise ValueError(f"{env_var} not found. Add it to your .env file.")

    token = raw_token.strip().strip('"').strip("'").strip()

    # Handles accidental values such as: VIKTOR_API_TOKEN=vktrpat_...
    if token.startswith(f"{env_var}="):
        token = token.split("=", 1)[1].strip().strip('"').strip("'").strip()

    # Handles accidental values such as: Authorization: Bearer vktrpat_...
    if token.lower().startswith("authorization:"):
        token = token.split(":", 1)[1].strip()

    # Handles accidental values such as: Bearer vktrpat_...
    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()

    token = token.strip().strip('"').strip("'").strip()

    if not token:
        raise ValueError(f"{env_var} is empty.")

    if not token.startswith("vktrpat_"):
        raise ValueError(
            f"{env_var} does not look like a VIKTOR Personal Access Token. "
            "It should start with 'vktrpat_'. Copy the PAT from your VIKTOR profile settings."
        )

    if any(ch.isspace() for ch in token):
        raise ValueError(
            f"{env_var} contains whitespace. Copy only the token value, without spaces or newlines."
        )

    return token


def mask_token(token: str) -> str:
    if len(token) <= 18:
        return token[:8] + "..."
    return token[:12] + "..." + token[-6:]


api_token = normalize_viktor_pat(os.getenv("VIKTOR_API_TOKEN"))

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {api_token}",
    "Accept": "application/json",
})

print(f"Using VIKTOR PAT: {mask_token(api_token)}")
print(f"Authorization header shape: Bearer {api_token[:8]}...")


def viktor_request(
    method: str,
    path: str,
    *,
    params: Dict[str, Any] | None = None,
    json_body: Dict[str, Any] | None = None,
) -> Dict[str, Any]:
    """Perform a VIKTOR REST API request and return JSON."""
    url = f"{BASE_URL}{path}"

    response = session.request(
        method=method,
        url=url,
        params=params,
        json=json_body,
        timeout=30,
    )

    if not response.ok:
        try:
            details = response.json()
        except ValueError:
            details = response.text

        raise RuntimeError(
            f"{method.upper()} {url} failed with HTTP {response.status_code}: {details}"
        )

    if not response.content:
        return {}

    try:
        return response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"{method.upper()} {url} did not return valid JSON: {response.text[:500]}"
        ) from exc


# Small connectivity check: list workspaces visible to the token.
workspaces = viktor_request("GET", "/workspaces/", params={"limit": 1, "offset": 0})
print("✓ Connected to VIKTOR REST API")
print(f"Workspace list response keys: {list(workspaces.keys())}")

## 2. Fetch Entity and Parametrization

In [ ]:
# Fetch the entity through REST
entity = viktor_request(
    "GET",
    f"/workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/",
    params={
        "properties": "true",     # include saved entity properties / params
        "clean_params": "true",   # return cleaned parameter values where supported
        "param_types": "true",
        "summary_types": "true",
    },
)

print(f"Entity: {entity.get('name', '<no name returned>')}")
print(f"Type: {entity.get('entity_type_name', entity.get('entity_type', '<no type returned>'))}")
print(f"Response keys: {list(entity.keys())}")

def extract_saved_params(entity_payload: Dict[str, Any]) -> Dict[str, Any]:
    """Extract saved params/properties from a VIKTOR REST entity response."""
    # The REST response schema exposes saved values as `properties`. Some API versions or SDK wrappers may name them `params`.
    for key in ("properties", "params", "last_saved_params"):
        value = entity_payload.get(key)
        if isinstance(value, dict):
            return value

    # Defensive fallback for versions that nest revision data.
    for key in ("last_saved_revision", "latest_revision", "revision"):
        value = entity_payload.get(key)
        if isinstance(value, dict) and isinstance(value.get("params"), dict):
            return value["params"]

    raise KeyError(
        "No saved params/properties found in the REST response. "
        f"Available keys: {list(entity_payload.keys())}"
    )

params = extract_saved_params(entity)
print(f"\nParameter keys: {list(params.keys()) if params else 'None'}")


## 3. Get Parametrization Definition

We'll extract the parametrization structure by analyzing the entity type and saved parameters.

In [ ]:
def infer_json_schema_from_params(params: dict) -> dict:
    """
    Infer JSON Schema from VIKTOR parametrization data.
    
    This is a basic inference - for accurate schemas, you should
    inspect the actual Parametrization class in the app code.
    """
    schema = {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
    
    for key, value in params.items():
        if isinstance(value, bool):
            schema["properties"][key] = {"type": "boolean"}
        elif isinstance(value, int):
            schema["properties"][key] = {"type": "integer"}
        elif isinstance(value, float):
            schema["properties"][key] = {"type": "number"}
        elif isinstance(value, str):
            schema["properties"][key] = {"type": "string"}
        elif isinstance(value, list):
            schema["properties"][key] = {
                "type": "array",
                "items": infer_type_from_value(value[0]) if value else {"type": "string"}
            }
        elif isinstance(value, dict):
            schema["properties"][key] = infer_json_schema_from_params(value)
        elif value is None:
            schema["properties"][key] = {"type": ["string", "null"]}
        
        # Mark as required if value is not None
        if value is not None:
            schema["required"].append(key)
    
    return schema

def infer_type_from_value(value):
    """Helper to infer JSON Schema type from a single value."""
    if isinstance(value, bool):
        return {"type": "boolean"}
    elif isinstance(value, int):
        return {"type": "integer"}
    elif isinstance(value, float):
        return {"type": "number"}
    elif isinstance(value, str):
        return {"type": "string"}
    elif isinstance(value, dict):
        return infer_json_schema_from_params(value)
    else:
        return {"type": "string"}

# Generate schema
if params:
    schema = infer_json_schema_from_params(params)
    print("✓ Schema generated")
    print(f"\nProperties: {list(schema['properties'].keys())}")
else:
    print("⚠ No parameters found - creating empty schema")
    schema = {
        "type": "object",
        "properties": {},
        "additionalProperties": False
    }

## 4. Preview the Schema

In [ ]:
print(json.dumps(schema, indent=2))

## 5. Save to JSON File

In [ ]:
output_path = Path(OUTPUT_FILE)
with open(output_path, "w") as f:
    json.dump(schema, f, indent=2)

print(f"✓ Schema saved to: {output_path.absolute()}")

## 6. Display Sample Parameters

In [ ]:
print("Sample parameters from entity:")
print(json.dumps(params, indent=2, default=str))